# Part 2: Normalize Data

---

## Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference

## Exploratory Data Analysis

**Exploratory Data Analysis (EDA)** is a vital diagnostic and discovery phase in RNA-seq workflows. It allows us to normalize datasets for meaningful cross-sample comparison and identify primary drivers of variation, which in turn helps generate biological hypotheses.

To master these techniques, we will utilize the previously saved `ccle_counts_subset` and `ccle_meta_subset` datasets. While the principles of EDA apply to the entire CCLE, we will focus specifically on lung cancer samples for this initial analysis to maintain a clear and manageable scope.

Core Objectives of EDA are:
1.  **Normalization:** Adjusting for sequencing depth and library size so expression levels are comparable across samples.
2.  **Dimensionality Reduction:** Using techniques like **PCA** (Principal Component Analysis) to visualize how samples cluster by lineage or mutation.
3.  **Quality Control:** Detecting outliers or batch effects that could skew downstream differential expression results.

## Read in Data

In [ ]:
ccle_counts_subset = pd.read_csv('outs/1_ccle_counts_subset.csv', header=0, index_col=0)
ccle_counts_subset

In [ ]:
ccle_meta_subset = pd.read_csv('outs/1_ccle_meta_subset.csv', header=0, index_col=0)
ccle_meta_subset

### Removing lowly expressed genes

To optimize computational efficiency and reduce stochastic noise, we will exclude lowly expressed genes from the analysis. By filtering out genes with a total count of less than 100 across all samples, we ensure that our downstream statistical models focus on genes with sufficient power for robust inference."

In [ ]:
keep = (ccle_counts_subset.sum(axis=1)>=100)
ccle_counts_subset_cleaned = ccle_counts_subset[keep]
ccle_counts_subset_cleaned
print(f"Total of {ccle_counts_subset.shape[0] - ccle_counts_subset_cleaned.shape[0]} genes are removed from the dds.")

## Data Normalization: Leveling the Analytical Playing Field

The first step in Exploratory Data Analysis (EDA) for RNA-seq is **normalization**. Without it, technical artifacts—rather than biological signals—will dominate our results. Normalization primarily addresses two major hurdles: **Sequencing Depth** and **Heteroscedasticity**.

### The Challenge of Sequencing Depth (Library Size)

Samples often vary significantly in their total number of sequenced reads, known as **sequencing depth**. 

Consider an experiment where a single lung cancer cell line sample, `DMS53_LUNG`, is split in two. **Sample A** is sequenced at $5$ million reads, while **Sample B** reaches $10$ million. Although they are biologically identical, Sample B will appear to have double the expression for every gene simply because it was sequenced more deeply. 

Here, to correct this, we use the **PyDESeq2** package, which avoids "reinventing the wheel" by implementing a two-step normalization process:

* **Size Factor Calculation:** PyDESeq2 estimates a scaling factor for each sample that reflects its relative library size.
* **Count Normalization:** Raw counts are divided by these size factors, mathematically removing the bias created by differing sequencing depths.

>**Note:** PyDESeq2 is an open-source Python re-implementation of the R-based DESeq2 package, allowing users to perform normalization, dispersion estimation, and statistical testing for differential expression using negative binomial generalized linear models. For more information, refer to [original paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC10502239/) and the [blog](https://www.owkin.com/blogs-case-studies/pydeseq2-a-new-tool-in-the-python-omics-ecosystem#:~:text=PyDESeq2%20is%20designed%20to%20be%20easy%20to,capabilities%20for%20efficient%20analysis%20of%20large%20datasets.) published by the package developers.


### Addressing Heteroscedasticity

In statistics, **heteroscedasticity** occurs when the variance of a measurement depends on its mean. In RNA-seq, highly expressed genes naturally exhibit much higher variance than lowly expressed genes. 

If left unaddressed, these "loud" genes will overwhelm our EDA, masking potentially critical biological signals from genes with lower expression levels. To "level the playing field," we need to decouple variance from the mean.

PyDESeq2 (and of-course R-based DESeq2) utilizes **Variance Stabilizing Transformation (VST)** to transform the data so that variance becomes roughly independent of the mean expression. In other words, VST adjusts the values so that the variance of lowly expressed genes becomes comparable to that of highly expressed genes. Once transformed, these counts are ideal for distance-based visualizations like **Principal Component Analysis (PCA)** and **Hierarchical Clustering**.

> **Note:** While VST-transformed data is perfect for visualization and clustering, it should **not** be used for differential expression analysis. We will use the raw counts and the negative binomial model for that specific task in later modules.

### Setting up `DeseqDataSet` object

Let's use PyDESeq2 package and create a DeseqDataSet data object that holds everything needed for further analysis.

`DeseqDataSet` class takes counts dataframe in `Number of samples` $\times$ `Number of genes` format. We will need to transpose the dataframe:

In [ ]:
ccle_counts_subset_cleaned_T = ccle_counts_subset_cleaned.transpose(copy=True).astype(int)
ccle_counts_subset_cleaned_T

In [ ]:
# 1. Initialize the computational "engine"
# This object manages how the math is solved, for example, using 4 CPU cores 
# to parallelize the calculations across thousands of genes.
# We can set it to `None` to use all available cpus.
inference = DefaultInference(n_cpus=None)

# 2. Create the DESeq2 Data Container
dds = DeseqDataSet(
    counts=ccle_counts_subset_cleaned_T,    # Matrix of raw RNA-seq counts (genes as columns, samples as rows)
    metadata=ccle_meta_subset,   # Dataframe containing sample info (must match counts_df index)
    design="~Pathology", # The statistical formula (e.g., comparing condition A vs condition B)
    refit_cooks=True,    # Automatically detect and re-run genes with extreme outliers
    inference=inference, # Pass the engine defined above to handle the optimization
    # n_cpus=8,          # Optional: can override the inference object's CPU count here
)

In [ ]:
dds

Before we can stabilize the variance, the model needs to understand the dispersion (noise) in our data.

In [ ]:
dds.deseq2()

In [ ]:
dds

Now, we can access to all computed parameters inside `dds` object. As an example, here is how we would access **size factors**, **normalized counts** (`normed_counts`), and **dispersion values** (`dispersions`):

In [ ]:
dds.obs['size_factors']

In [ ]:
dds.layers['normed_counts']

In [ ]:
dds.var['dispersions']

### Variance Stabilizing Transformation (VST)

We will use `vst()` to carry out variance stabilizing transformation. It transforms the counts data to stabilize variance across the entire spectrum of gene expression levels.

The `vst()` method has a few important arguments that control how the math is handled:

* **`use_design` (default: `False`)**: 
    * If `True`, it uses the design formula (e.g., `~condition`) to estimate the trend. 
    * If `False` (Blind), it ignores the design. **"Blind" is recommended for EDA/PCA** because it ensures that the visualization isn't biased by the groups you are trying to discover.
* **`fit_type`**: Usually left as `'parametric'`, which fits a dispersion-mean relationship curve.

In [ ]:
dds.vst()

In [ ]:
dds.layers['vst_counts']

### Visualizing Variance Stabilization

#### 1. Variance vs. Mean

To evaluate the transformation, we will compare **Mean-Variance Scatter Plots** for raw versus stabilized counts.

Plotting raw counts reveals a strong positive correlation between mean expression and variance. This **heteroscedasticity** is a technical artifact where high-abundance genes exhibit disproportionately high "noise," potentially overwhelming biological signals.

After applying **VST**, the trend "flattens" significantly. The variance becomes roughly constant—or **homoscedastic**—across the entire expression spectrum. 

By decoupling variance from the mean, VST ensures that genes are weighted by their biological significance rather than their absolute abundance. This is a critical prerequisite for distance-based analyses like **PCA** and **Hierarchical Clustering**.

In [ ]:
# Create raw counts in pandas dataframe
raw_df = ccle_counts_subset_cleaned_T.copy()

# Creat vst counts in pandas dataframe
vst_df = pd.DataFrame(
    dds.layers['vst_counts'],
    index=dds.obs_names,
    columns=dds.var_names
)

# Create a dataframe with mean and variance of genes across samples of raw counts.
raw_mean_var_df = pd.concat([raw_df.mean(axis=0), raw_df.var(axis=0)], axis=1)
raw_mean_var_df.columns = ['Mean', 'Variance']
raw_mean_var_df.index.name = 'Gene'

# Create a dataframe with mean and variance of genes across samples of vst counts.
vst_mean_var_df = pd.concat([vst_df.mean(axis=0), vst_df.var(axis=0)], axis=1)
vst_mean_var_df.columns = ['Mean', 'Variance']
vst_mean_var_df.index.name = 'Gene'

In [ ]:
# Create scatter plots for mean vs variance
fig, axes = plt.subplots(1, 2, figsize = (12, 5))

# Raw counts subplot
sns.scatterplot(ax=axes[0], data=raw_mean_var_df, x='Mean', y='Variance', marker='o', alpha=0.5, s=1, color='k')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel("log10(Gene counts mean)")
axes[0].set_ylabel("log10(Gene counts variance)")
axes[0].set_title("Raw counts")

# Variance-stablized counts subpot
sns.scatterplot(ax=axes[1], data=vst_mean_var_df, x='Mean', y='Variance', marker='o', alpha=0.5, s=1, color='k')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel("log10(Gene counts mean)")
axes[1].set_ylabel("log10(Gene counts variance)")
axes[1].set_title("Variance-stablized counts");

As shown, the Variance Stabilizing Transformation (VST) effectively mitigates heteroscedasticity, decoupling the variance from the mean expression levels to produce a more homoscedastic distribution across the gene space.

#### 2. Expressoin of high vs low expressed genes

To better understand the effect of VST, we can visualize the data distributions in another way.

In raw RNA-seq data, variance is tied to the mean. High-abundance genes exhibit massive fluctuations, while low-abundance genes appear virtually "flat." This skewness means a handful of highly expressed genes will dominate any distance-based analysis (like PCA).

After **VST**, the "spread" of each gene becomes comparable, regardless of its average expression. The transformation effectively squashes the variance of high-abundance genes and stretches the variance of low-abundance genes, "leveling the playing field."

In [ ]:
# Retrieve highly expressed genes.
low_percentile_top = np.quantile(raw_mean_var_df['Mean'], 0.799)
high_percentile_top = np.quantile(raw_mean_var_df['Mean'], 0.801)
high_genes = raw_mean_var_df[(raw_mean_var_df['Mean'] >= low_percentile_top) & (raw_mean_var_df['Mean'] <= high_percentile_top)]
high_genes_names = high_genes.index.values

# Retrieve lowly expressed genes
low_percentile_bot = np.quantile(raw_mean_var_df['Mean'], 0.199)
high_percentile_bot = np.quantile(raw_mean_var_df['Mean'], 0.201)
low_genes = raw_mean_var_df[(raw_mean_var_df['Mean'] >= low_percentile_bot) & (raw_mean_var_df['Mean'] <= high_percentile_bot)]
low_genes_names = low_genes.index.values

In [ ]:
top_df = pd.DataFrame({
    'Gene': np.tile(high_genes_names, reps=raw_df.shape[0]),
    'Raw_counts': np.ravel(raw_df[high_genes_names].to_numpy()),
    'Vst_counts': np.ravel(vst_df[high_genes_names].to_numpy()),
})

top_df_long = pd.melt(
    top_df,
    id_vars='Gene',
    value_vars=top_df.columns,
    var_name='Count type',
    value_name='Expression'
)
top_df_long

In [ ]:
bot_df = pd.DataFrame({
    'Gene': np.tile(low_genes_names, reps=raw_df.shape[0]),
    'Raw_counts': np.ravel(raw_df[low_genes_names].to_numpy()),
    'Vst_counts': np.ravel(vst_df[low_genes_names].to_numpy()),
})

bot_df_long = pd.melt(
    bot_df,
    id_vars='Gene',
    value_vars=bot_df.columns,
    var_name='Count type',
    value_name='Expression'
)
bot_df_long

In [ ]:
fig , axes = plt.subplots(1, 2, figsize=(12, 5))

# Highly expressed genes
sns.boxplot(ax=axes[0], data=top_df_long, x='Count type', y='Expression', showfliers=False, palette='Set2')
sns.stripplot(
    ax=axes[0],
    data=top_df_long, 
    x='Count type', 
    y='Expression', 
    jitter=True,     
    size=2,          
    alpha=0.1,       
    color='black'
)
axes[0].set_yscale('log')
axes[0].set_ylim(0.05, 1e5)
axes[0].set_title('Highly expressed genes')
axes[0].set_ylabel('log10(Expression)')

# Lowly expressed genes
sns.boxplot(ax=axes[1], data=bot_df_long, x='Count type', y='Expression', showfliers=False, palette='Set2')
sns.stripplot(
    ax=axes[1],
    data=bot_df_long, 
    x='Count type', 
    y='Expression', 
    jitter=True,     
    size=2,          
    alpha=0.1,       
    color='black'
)
axes[1].set_yscale('log')
axes[1].set_ylim(0.05, 1e5)
axes[1].set_title('Lowly expressed genes')
axes[1].set_ylabel('log10(Expression)');

VST mitigates the mean-variance dependency inherent in RNA-seq data. By disproportionately compressing the scales of highly expressed genes while inflating those of low-abundance features, VST produces a **homoscedastic** dataset.

As illustrated in the boxplots, the stabilized counts exhibit a significantly "squashed" variance, with expression levels across the entire transcriptome centered around a similar magnitude (log-scale $\approx 10$). This effectively **"levels the playing field,"** preventing high-count genes from dominating the statistical signal and allowing subtle biological patterns from lowly expressed genes to be captured during **Principal Component Analysis (PCA)** and **Hierarchical Clustering**.